# M05 locked CPU qualification
Select **Runtime → Change runtime type → no accelerator** first. Run cells in order.
CPU results from this notebook qualify only the actual recorded Colab host/profile.
A saved notebook does not preserve its runtime, model, checkpoints or journals.
No binding, density, pressure/virial or molecular accuracy claim is made.

The baseline source pin below reproduces the published checkpoint. For extension
experiments, set an exact reviewed implementation commit explicitly. Keep all
scientific work on local `/content`; export only at stopped, committed safe points.
Authentication, runtime selection and durable storage authorization belong to you.


In [ ]:
from pathlib import Path
# Exact published baseline; replace explicitly with the reviewed implementation SHA
# only when running the extension cells below. A branch name is never accepted.
SOURCE_SHA = "7b41213e87bfcc3ce76def7ffb64565110bbafec"
REPOSITORY_URL = "https://github.com/georgpou/AToM_MLMM.git"
REPO = Path("/content/AToM_MLMM")
SETUP = Path("/content/atom-mlmm-colab-cpu")
# New output for each runtime/source experiment; never reuse/overwrite an attempt.
EVIDENCE = Path("/content/m05-evidence-baseline-01")
BASELINE_SHA = "7b41213e87bfcc3ce76def7ffb64565110bbafec"
MODEL_SHA = "165cce4cfec5a34b9c64d4ebf95de15d71106bb584b7291c8470f0749977c46f"
assert len(SOURCE_SHA)==40 and all(c in "0123456789abcdef" for c in SOURCE_SHA)
assert not EVIDENCE.exists(), "Use a new evidence directory; preserve earlier results."


In [ ]:
import datetime, json, os, signal, subprocess, time

def run_command(argv, label, timeout, cwd=None, env=None):
    """Bound a subprocess; retain full logs and exit status, including failures."""
    EVIDENCE.mkdir(parents=True, exist_ok=True)
    log = EVIDENCE / (label + ".log")
    status = EVIDENCE / (label + "-command.json")
    assert not log.exists() and not status.exists(), "Command label already used."
    started = time.monotonic()
    record = {"argv":list(map(str,argv)), "cwd":str(cwd or REPO),
              "utc":datetime.datetime.now(datetime.timezone.utc).isoformat(),
              "timeout_seconds":timeout, "timed_out":False, "exit_code":None}
    with log.open("w") as output:
        process = subprocess.Popen(argv, cwd=cwd or REPO, env=env,
                                   stdout=output, stderr=subprocess.STDOUT,
                                   start_new_session=True)
        try:
            record["exit_code"] = process.wait(timeout=timeout)
        except subprocess.TimeoutExpired:
            record["timed_out"] = True
            os.killpg(process.pid, signal.SIGKILL)
            record["exit_code"] = process.wait()
            raise
        finally:
            record["elapsed_seconds"] = time.monotonic()-started
            status.write_text(json.dumps(record,indent=2)+"\n")
    print(label, "exit", record["exit_code"], "log:",log)
    if record["exit_code"]:
        raise subprocess.CalledProcessError(record["exit_code"],argv)

# Every computation is a fresh Bash subprocess with explicit main activation.
import shlex
def cpu_command(command, label, timeout=900):
    prefix = "set -euo pipefail; source " + shlex.quote(str(SETUP/"activate.sh"))
    prefix += '; export OPENBLAS_NUM_THREADS=2 OMP_NUM_THREADS=2 PYTHONPATH="$PWD/src"; '
    return run_command(["bash","-c",prefix+command],label,timeout)


In [ ]:
import hashlib, platform, shutil
assert platform.system()=="Linux" and platform.machine()=="x86_64", "Pinned CPU locks require Linux x86_64."
assert shutil.disk_usage("/content").free >= 16*1024**3, "Installation requires at least 16 GiB free disk."
created_checkout = not REPO.exists()
if created_checkout:
    run_command(["git","clone","--no-checkout",REPOSITORY_URL,str(REPO)],"clone",600,cwd="/content")
assert subprocess.check_output(["git","remote","get-url","origin"],cwd=REPO,text=True).strip()==REPOSITORY_URL
if not created_checkout:
    assert not subprocess.check_output(["git","status","--porcelain"],cwd=REPO,text=True).strip(), "Preserve local changes; use a fresh checkout."
run_command(["git","fetch","origin",SOURCE_SHA],"fetch-source",600)
run_command(["git","checkout","--detach",SOURCE_SHA],"checkout-source",60)
actual_sha=subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO,text=True).strip()
assert actual_sha==SOURCE_SHA
assert hashlib.sha256((REPO/"models/mace-off23-small/MACE-OFF23_small.model").read_bytes()).hexdigest()==MODEL_SHA
run_command(["bash","-c","cd environment/cloud-cpu && sha256sum -c bundle.sha256"],"verify-install-bundle",120)
(EVIDENCE/"source-model.json").write_text(json.dumps({"source_sha":actual_sha,"model_sha256":MODEL_SHA,
    "notebook_revision":"m05-cpu-v1", "runtime":"Colab CPU selected by user"},indent=2)+"\n")


In [ ]:
import sys
resource_record={"utc":datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "os":platform.platform(),"architecture":platform.machine(),"kernel_python":sys.version,
    "logical_processors":os.cpu_count(),"meminfo":Path("/proc/meminfo").read_text(),
    "disk_bytes":dict(zip(("total","used","free"),shutil.disk_usage("/content"))),
    "cpu_threads":2,"source_sha":SOURCE_SHA,"model_sha256":MODEL_SHA,
    "baseline_real_atoms":{"abfe":56,"rbfe":65},"baseline_workers":1,
    "seed":41,"preparation_steps":6,"sampling_steps_per_control":18,
    "wall_time_limit_seconds":900,"persistence_interval":"every complete sample/round; export after each stopped command"}
(EVIDENCE/"resource-preflight.json").write_text(json.dumps(resource_record,indent=2)+"\n")
print({k:resource_record[k] for k in ("os","logical_processors","disk_bytes","cpu_threads")})


In [ ]:
run_command(["bash","environment/cloud-cpu/install.sh"],"install-unchanged-locks",1800,
            env={**os.environ,"ATOM_MLMM_SETUP_ROOT":str(SETUP)})
cpu_command("python "+shlex.quote(str(SETUP/"validate.py"))+" --repository "+shlex.quote(str(REPO)), "strict-setup",900)
# Record exact packages and actual OpenMM platforms separately from the Colab kernel.
cpu_command("python -m pip freeze && python -c 'import sys,openmm as m,torch; print(sys.executable); print([m.Platform.getPlatform(i).getName() for i in range(m.Platform.getNumPlatforms())]); print(torch.__version__,torch.get_default_dtype())'", "installed-profile",120)
# Full meaningful locked CPU suite; no skips are silently counted as passes.
cpu_command("python -m pytest -q", "cpu-suite",1800)


In [ ]:
for kind in ("abfe","rbfe"):
    config=REPO/"fixtures/solvated_fragment/v1"/kind/"config.json"
    cpu_command("python -m atm_mlmm check "+shlex.quote(str(config)),kind+"-check",120)
    output=EVIDENCE/(kind+"-baseline")
    cpu_command("python -m atm_mlmm run "+shlex.quote(str(config))+" --output "+shlex.quote(str(output))+" --trusted",kind+"-baseline-run",900)
# Each attempt contains raw energies, all-real forces, live box, preparation,
# immutable sample States/checkpoints, verified worker/model/source assets.


## Extension experiments (only after baseline evidence)
Save/export the baseline first. Set `SOURCE_SHA` to the exact reviewed M05 source
commit, use a fresh checkout/evidence directory, and repeat setup/profile checks.
The default baseline pin deliberately has no dense/exchange extension commands.
Before heavier work, record actual atom/worker counts, remaining RAM/disk, seeds,
steps, maximum wall time and export interval. Start with the measured small pilot.


In [ ]:
assert SOURCE_SHA != BASELINE_SHA, "Run extension cells only with the explicit reviewed M05 implementation commit."
assert (REPO/"src/atm_mlmm/exchange.py").is_file()
cpu_command("python -m pytest tests/workflow/test_dense_solvent.py tests/workflow/test_solvated_process_restart.py tests/workflow/test_persistent_exchange.py -q", "extension-numerical-restart-preflight",1800)
# Tests establish independent both-map full forces, FD sweeps, offline portability,
# checkpoint vs portable-State semantics and round failures before larger pilots.


In [ ]:
extension_resources={**resource_record,"source_sha":SOURCE_SHA,"controls_real_atoms":{"abfe":224,"rbfe":233},
    "exchange_workers":2,"exchange_seed":73,"exchange_rounds":3,"steps_per_round":2,
    "wall_time_limit_seconds":900,"persistence_interval":"one complete two-worker round"}
(EVIDENCE/"extension-resource-preflight.json").write_text(json.dumps(extension_resources,indent=2)+"\n")
for kind in ("abfe","rbfe"):
    config=REPO/"fixtures/solvated_fragment/v2"/kind/"config.json"
    output=EVIDENCE/(kind+"-denser")
    cpu_command("python -m atm_mlmm check "+shlex.quote(str(config)),kind+"-denser-check",120)
    cpu_command("python -m atm_mlmm run "+shlex.quote(str(config))+" --output "+shlex.quote(str(output))+" --trusted",kind+"-denser-run",900)
    exchange=EVIDENCE/(kind+"-exchange")
    cpu_command("python -m atm_mlmm exchange "+shlex.quote(str(output))+" --output "+shlex.quote(str(exchange))+" --rounds 3 --steps-per-round 2 --seed 73 --trusted",kind+"-exchange-run",900)


## Save before disconnect
Stop active calculations before copying. Download complete attempt directories,
not just summaries/notebooks. The next cell exports a hash-verified snapshot using
the reviewed M05 implementation. Choose a user-authorized durable destination
(e.g. a manually mounted Drive directory); a local `/content` copy alone is lost
with the runtime. Copy and verify before deleting anything. Keep secrets outside
these scientific directories. Drive is export-only, never the active journal.

The baseline can be downloaded as complete directory archives from Colab's Files
panel, preserving all files. Verify the returned archive with the M05 transport
before calling it retained evidence. No Drive mount/authentication is automatic.


In [ ]:
assert SOURCE_SHA != BASELINE_SHA, "Use the reviewed M05 export helper; baseline assets remain unchanged."
# Set to an unused directory in durable storage you have authorized/mounted.
DURABLE_DESTINATION = None
assert DURABLE_DESTINATION is not None, "Select and authorize a durable destination or download complete attempts."
# This subprocess closes its log before the evidence tree is copied.
# Export each stopped attempt independently, keeping command logs outside it.
for name in ("abfe-baseline","rbfe-baseline","abfe-denser","rbfe-denser","abfe-exchange","rbfe-exchange"):
    source=EVIDENCE/name
    if source.exists():
        script="from atm_mlmm.evidence import export_evidence; import sys; export_evidence(sys.argv[1],sys.argv[2])"
        destination=Path(DURABLE_DESTINATION)/name
        cpu_command("python -c "+shlex.quote(script)+" "+shlex.quote(str(source))+" "+shlex.quote(str(destination)),"export-"+name,600)


## Resume after interruption
Restore a complete verified export to fresh local `/content` storage, verify its
`.evidence.json`, and use the attempt's `worker/runtime/source` in `PYTHONPATH`.
Activate the identical locked CPU profile in every subprocess. Run
`python -m atm_mlmm resume /content/restored-attempt --trusted` for fixed windows,
or `python -m atm_mlmm resume-exchange /content/restored-exchange --trusted`.
Preserve incomplete transactions: normal resume rejects them. Inspect failures
before explicitly requesting `--recover-pending` for an exchange rollback/replay.
The export seal describes the pre-resume snapshot; keep that immutable snapshot
separate and resume a working copy with its transport seal removed.

Checkpoint continuation uses the same admitted profile. Portable States support
fixed-coordinate observables and make no promise of bitwise RNG/trajectory
continuation across changed hosts/hardware/libraries. Send returned complete
manifests, command exits/logs, raw records, checkpoints and resource reports for
verification. Notebook delivery/local validation alone is not a Colab pass.
